# Agentic GitHub Benchmark (kaggle-benchmarks / `kbench`)\n\n"
"Каждая модель через реальный GitHub REST API:\n"
"1. Создаёт свою ветку в вашем репозитории.\n"
"2. Выолняет задачу с собственным описанием возможностей.\n"
"3. Коммитит его в свою ветку.\n\n"
"Результат: лидерборд по успеху/токенам/стоимости + список созданных веток.\n\n"
"**Перед первым запуском на реальном репозитории — прочитайте раздел 3 "
"(токен и безопасность) и лучше протестируйте на одноразовом/тестовом "
"репозитории, а не на боевом.**

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import requests
import base64
import functools
import glob
import json
import os
import re
import sys
import zipfile
from datetime import datetime

kbench.config.enable_console_mode(quiet=False)  # человекочитаемый лог по ходу выполнения

# Фиксируем реальный корень ноутбука ОДИН РАЗ — все пути (архив, run.json) считаются
# от него, а не от "текущего" cwd (который может уехать, если где-то в коде — в т.ч.
# у вас — есть os.chdir()).
NOTEBOOK_ROOT = os.path.abspath(os.getcwd())

## 1. Модели для сравнения

In [ ]:
models_dict = kbench.llms
if not models_dict:
    raise RuntimeError("В kbench.llms не обнаружено доступных моделей.")

print("--- Доступные модели ---")
for i, model_id in enumerate(models_dict.keys(), start=1):
    print(f"{i}. {model_id}")

In [ ]:
MODELS = [
    "anthropic/claude-opus-5@default",
    "xai/grok-4.20-0309-reasoning",
    "google/gemini-3.5-flash-lite",
    "openai/gpt-5.4-nano-2026-03-17",
    "xai/grok-4.20-0309-non-reasoning",
    # добавьте/уберите модели, доступные вашему proxy-токену
]

"## 2. GitHub: репозиторий, токен, безопасность\n\n"
"**Токен НЕ вставляйте текстом в ячейку.** Сохранённая версия ноутбука (и тем "
"более публичная) унесёт его с собой. Используйте Kaggle Secrets: "
"меню ноутбука → **Add-ons → Secrets** → добавьте секрет с именем `GITHUB_TOKEN`.\n\n"
"Рекомендации к самому токену:\n"
"- Fine-grained PAT, а не classic — можно ограничить одним конкретным репозиторием.\n"
"- Права только **Contents: Read and write** (и **Metadata: Read** — обычно "
"проставляется автоматически). Больше ничего не нужно.\n"
"- Протестируйте сначала на отдельном тестовом репозитории — скрипт создаёт "
"реальные ветки и коммиты, отката/удаления в коде нет (GitHub это позволяет "
"сделать вручную через UI, если веток наплодится больше, чем нужно)."

In [ ]:
# --- Репозиторий ---
GITHUB_OWNER = "mlaa4ml"
GITHUB_REPO = "KaggleModelsRepo"
GITHUB_BASE_BRANCH = "main"  # ветка, от которой создаются ветки моделей (только читаем из неё)

# --- Токен: только через Kaggle Secrets ---
try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as e:
    GITHUB_TOKEN = None
    print(
        "Не удалось получить GITHUB_TOKEN из Kaggle Secrets "
        f"({e}). Добавьте секрет с этим именем через Add-ons -> Secrets."
    )

assert GITHUB_TOKEN, "Нужен GITHUB_TOKEN (см. ячейку выше) для работы с GitHub API"

GITHUB_API = "https://api.github.com"
GITHUB_HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

# Быстрая проверка доступа и прав токена, прежде чем тратить деньги на моделей.
_check = requests.get(f"{GITHUB_API}/repos/{GITHUB_OWNER}/{GITHUB_REPO}", headers=GITHUB_HEADERS, timeout=15)
if _check.status_code != 200:
    raise RuntimeError(
        f"Нет доступа к {GITHUB_OWNER}/{GITHUB_REPO}: {_check.status_code} {_check.text[:300]}"
    )
print(f"Доступ к репозиторию {GITHUB_OWNER}/{GITHUB_REPO} подтверждён.")

"## 3. GitHub-инструменты (реальный REST API)\n\n"
"`create_branch` и `write_file` привязаны к ветке конкретной модели через "
"замыкание в `make_github_tools()` (см. раздел 6) — модель не может выбрать "
"произвольное имя ветки или писать в чужую/базовую ветку, даже если "
"попытается. `read_file` всегда читает только из `GITHUB_BASE_BRANCH`."

In [ ]:
def _gh_request(method: str, path: str, **kwargs) -> requests.Response:
    return requests.request(method, f"{GITHUB_API}{path}", headers=GITHUB_HEADERS, timeout=30, **kwargs)


def gh_list_branches() -> str:
    """List branch names currently in the repository."""
    resp = _gh_request("GET", f"/repos/{GITHUB_OWNER}/{GITHUB_REPO}/branches", params={"per_page": 100})
    if resp.status_code != 200:
        return f"ERROR: list_branches failed ({resp.status_code}): {resp.text[:300]}"
    return ", ".join(b["name"] for b in resp.json())


def gh_create_branch(branch_name: str) -> str:
    """Create a branch from GITHUB_BASE_BRANCH if it doesn't already exist (idempotent)."""
    exists = _gh_request("GET", f"/repos/{GITHUB_OWNER}/{GITHUB_REPO}/git/ref/heads/{branch_name}")
    if exists.status_code == 200:
        return f"Ветка {branch_name} уже существует."

    base = _gh_request("GET", f"/repos/{GITHUB_OWNER}/{GITHUB_REPO}/git/ref/heads/{GITHUB_BASE_BRANCH}")
    if base.status_code != 200:
        return f"ERROR: не нашёл базовую ветку {GITHUB_BASE_BRANCH} ({base.status_code}): {base.text[:300]}"
    base_sha = base.json()["object"]["sha"]

    created = _gh_request("POST", f"/repos/{GITHUB_OWNER}/{GITHUB_REPO}/git/refs", json={
        "ref": f"refs/heads/{branch_name}",
        "sha": base_sha,
    })
    if created.status_code not in (200, 201):
        return f"ERROR: create_branch failed ({created.status_code}): {created.text[:300]}"
    return f"Ветка {branch_name} создана от {GITHUB_BASE_BRANCH}."


def gh_read_file(path: str, branch: str) -> str:
    """Read a file's content from the given branch."""
    resp = _gh_request(
        "GET", f"/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents/{path}", params={"ref": branch}
    )
    if resp.status_code != 200:
        return f"ERROR: {path} not found on branch {branch} ({resp.status_code})"
    data = resp.json()
    if data.get("encoding") != "base64":
        return f"ERROR: unexpected encoding {data.get('encoding')!r} for {path}"
    return base64.b64decode(data["content"]).decode("utf-8", errors="replace")


def gh_write_file(path: str, content: str, branch: str) -> str:
    """Create or update a file with full content on the given branch (one commit)."""
    existing = _gh_request(
        "GET", f"/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents/{path}", params={"ref": branch}
    )
    sha = existing.json().get("sha") if existing.status_code == 200 else None

    payload = {
        "message": f"{path}: {'обновление' if sha else 'создание'} от {branch}",
        "content": base64.b64encode(content.encode("utf-8")).decode("ascii"),
        "branch": branch,
    }
    if sha:
        payload["sha"] = sha

    resp = _gh_request("PUT", f"/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents/{path}", json=payload)
    if resp.status_code not in (200, 201):
        return f"ERROR: write_file failed ({resp.status_code}): {resp.text[:300]}"
    commit_sha = resp.json().get("commit", {}).get("sha", "")[:7]
    return f"Записано {len(content)} символов в {path} (ветка {branch}), commit {commit_sha}"

"## 4. Раундовый движок с бюджетом шагов\n\n"
"Ограничивает и число вызовов инструментов за раунд, и число раундов на "
"модель — предсказуемый потолок стоимости и количества реальных API-вызовов "
"к GitHub на модель×задачу, независимо для каждой модели (бюджет не общий)."

In [ ]:
class StepBudget:
    """Общий счётчик вызовов инструментов на один раунд."""

    def __init__(self, limit: int):
        self.limit = limit
        self.used = 0

    def consume(self) -> bool:
        if self.used >= self.limit:
            return False
        self.used += 1
        return True

    @property
    def remaining(self) -> int:
        return max(0, self.limit - self.used)


def with_budget(tool_fn, budget: StepBudget):
    """Оборачивает инструмент, чтобы он считался в бюджет раунда.

    @functools.wraps(tool_fn) обязателен: без него у wrapped() сигнатура
    "(*args, **kwargs)", kbench не может построить из неё схему параметров
    для function calling и подсовывает моделям generic-схему {"args":...,
    "kwargs":...} — модели гадают вслепую и жгут бюджет на угадывание, а не
    на задачу (реальный баг, который был здесь раньше).
    """
    @functools.wraps(tool_fn)
    def wrapped(*args, **kwargs):
        if not budget.consume():
            return (
                f"ERROR: бюджет шагов на этот раунд исчерпан ({budget.limit}). "
                "Дальнейшие вызовы инструментов в этом раунде недоступны — "
                "заверши раунд текстовым ответом (кратко: что сделано, что нет)."
            )
        return tool_fn(*args, **kwargs)

    return wrapped


STEPS_PER_ROUND = 10   # вызовов инструментов за раунд
MAX_ROUNDS = 3         # раундов на пару модель×задача


def run_rounds(chat, llm, make_round_tools, task_check_fn, task_context: str):
    """Раундовый цикл поверх уже открытого kbench chat.

    make_round_tools(budget) -> list[callable]   свежий набор инструментов на раунд.
    task_check_fn() -> bool                      решена ли задача уже сейчас.
    Возвращает (solved, rounds_used, self_report).
    """
    self_report = None

    for round_num in range(1, MAX_ROUNDS + 1):
        is_last_round = round_num == MAX_ROUNDS
        budget = StepBudget(STEPS_PER_ROUND)
        tools = make_round_tools(budget)

        if round_num == 1:
            message = (
                f"{task_context}\\n\\n"
                f"У тебя есть до {STEPS_PER_ROUND} вызовов инструментов на этот раунд "
                f"и до {MAX_ROUNDS} раундов всего. Если не уложишься в этот раунд — "
                "продолжишь в следующем, история сохранится."
            )
        else:
            message = (
                f"Раунд {round_num}/{MAX_ROUNDS}. Новый бюджет: до {STEPS_PER_ROUND} "
                "вызовов инструментов. Продолжай с того места, где остановился(-ась)."
            )
        if is_last_round:
            message += (
                "\\n\\nЭТО ПОСЛЕДНИЙ ДОСТУПНЫЙ РАУНД. Сделай что успеешь в рамках "
                "бюджета, а в конце текстового ответа обязательно добавь блок:\\n"
                "===SELF_REPORT===\\n"
                "готово_процентов: <0-100>\\n"
                "осталось_сделать: <кратко>\\n"
                "нужно_ещё_шагов: <число>\\n"
                "===END_SELF_REPORT==="
            )

        response = llm.prompt(message, tools=tools)

        report_match = re.search(r"===SELF_REPORT===(.*?)===END_SELF_REPORT===", response, re.DOTALL)
        if report_match:
            self_report = report_match.group(1).strip()

        if task_check_fn():
            return True, round_num, self_report

    return False, MAX_ROUNDS, self_report

"## 5. Задача: реальная ветка + файл с заданием\n\n"
"`create_branch`/`write_file` жёстко привязаны к `branch_name` (= slug "
"модели) через замыкание в `make_github_tools` — их сигнатура для модели "
"либо без аргументов, либо только `path`/`content`, имя ветки модель не "
"передаёт и повлиять на него не может."

In [ ]:
def slugify_model_id(model_id: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", model_id.lower()).strip("-")


def make_github_tools(branch_name: str, budget: StepBudget):
    def list_branches() -> str:
        """List branch names currently in the repository."""
        return gh_list_branches()

    def create_branch() -> str:
        """Create your assigned branch in the repository (idempotent, safe to call more than once)."""
        return gh_create_branch(branch_name)

    def read_file(path: str) -> str:
        """Read a file's content from the base branch, for context."""
        return gh_read_file(path, branch=GITHUB_BASE_BRANCH)

    def write_file(path: str, content: str) -> str:
        """Create or update a file with FULL new content on your assigned branch."""
        return gh_write_file(path, content, branch=branch_name)

    return [
        with_budget(list_branches, budget),
        with_budget(create_branch, budget),
        with_budget(read_file, budget),
        with_budget(write_file, budget),
    ]


GITHUB_TASK_SYSTEM_PROMPT = (
    "Ты — ассистент, который работает с реальным GitHub-репозиторием ТОЛЬКО "
    "через предоставленные инструменты. Ты не можешь исполнять произвольный "
    "код или обращаться к GitHub напрямую — только list_branches, "
    "create_branch, read_file, write_file."
)

GITHUB_TASK_USAGE_LOG = []
GITHUB_TASK_ROUNDS_LOG = []


@kbench.task(name="github_repo_capabilities")
def github_repo_capabilities_eval(llm, model_id: str) -> bool:
    branch_name = slugify_model_id(model_id)
    cap_path = f"CAPABILITIES_{branch_name}.md"

    task_context = (
        f"Репозиторий: {GITHUB_OWNER}/{GITHUB_REPO} (базовая ветка: {GITHUB_BASE_BRANCH}).\\n"
        "У тебя есть инструменты: list_branches, create_branch, read_file, write_file.\\n"
        "Задача:\\n"
        "1. Описание задачи.\\n"
        "Существующие файлы проекта можно прочитать через read_file для контекста."
    )

    def make_round_tools(budget):
        return make_github_tools(branch_name, budget)

    def task_check_fn() -> bool:
        content = gh_read_file(cap_path, branch=branch_name)
        return not content.startswith("ERROR:") and len(content.strip()) > 40

    with kbench.chats.new(f"{model_id}::github_repo_capabilities") as chat:
        kbench.user.send(GITHUB_TASK_SYSTEM_PROMPT)
        solved, rounds_used, self_report = run_rounds(
            chat, llm, make_round_tools, task_check_fn, task_context
        )

    usage = chat.usage
    GITHUB_TASK_USAGE_LOG.append({
        "model_id": model_id,
        "input_tokens": usage.input_tokens or 0,
        "output_tokens": usage.output_tokens or 0,
        "input_cost_usd": (usage.input_tokens_cost_nanodollars or 0) / 1e9,
        "output_cost_usd": (usage.output_tokens_cost_nanodollars or 0) / 1e9,
        "latency_ms": usage.total_backend_latency_ms,
    })
    GITHUB_TASK_ROUNDS_LOG.append({
        "model_id": model_id,
        "branch": branch_name,
        "capabilities_file": cap_path,
        "rounds_used": rounds_used,
        "max_rounds": MAX_ROUNDS,
        "solved": solved,
        "self_report": self_report,
    })

    if not solved:
        kbench.assertions.assert_fail(
            expectation=f"Файл {cap_path} не появился в ветке {branch_name} за {MAX_ROUNDS} раундов"
        )
    return solved

## 6. Прогон по всем моделям

In [ ]:
github_task_completed = []

for model_id in MODELS:
    llm = kbench.llms[model_id]
    eval_df = pd.DataFrame([{"model_id": model_id}])

    results = github_repo_capabilities_eval.evaluate(
        llm=[llm],
        evaluation_data=eval_df,
        on_failure="continue",   # одна упавшая модель не рушит весь прогон
        max_attempts=1,          # раунды уже дают модели incrementally больше шансов
    )

    completed = results.completed_runs.as_dataframe()
    completed["model_id"] = model_id
    github_task_completed.append(completed)

    print(f"{model_id}: {len(results.completed_runs)} completed, {len(results.errored_runs)} errored")

github_task_runs_df = pd.concat(github_task_completed, ignore_index=True)
github_task_usage_df = pd.DataFrame(GITHUB_TASK_USAGE_LOG)
github_task_rounds_df = pd.DataFrame(GITHUB_TASK_ROUNDS_LOG)

## 7. Итоги: успех × раунды × токены × стоимость

In [ ]:
pass_rate = github_task_runs_df.set_index("model_id")["result"].rename("solved")

usage_agg = github_task_usage_df.groupby("model_id").agg(
    input_tokens=("input_tokens", "sum"),
    output_tokens=("output_tokens", "sum"),
    input_cost_usd=("input_cost_usd", "sum"),
    output_cost_usd=("output_cost_usd", "sum"),
)
usage_agg["total_cost_usd"] = usage_agg["input_cost_usd"] + usage_agg["output_cost_usd"]
usage_agg["cost_per_1m_input_usd"] = (
    usage_agg["input_cost_usd"] / usage_agg["input_tokens"].replace(0, pd.NA) * 1_000_000
)
usage_agg["cost_per_1m_output_usd"] = (
    usage_agg["output_cost_usd"] / usage_agg["output_tokens"].replace(0, pd.NA) * 1_000_000
)

leaderboard = (
    pass_rate.to_frame()
    .join(usage_agg)
    .join(github_task_rounds_df.set_index("model_id")[["branch", "capabilities_file", "rounds_used"]])
)
leaderboard = leaderboard.sort_values(["solved", "total_cost_usd"], ascending=[False, True])

totals = pd.Series({
    "solved": leaderboard["solved"].sum(),
    "input_tokens": leaderboard["input_tokens"].sum(),
    "output_tokens": leaderboard["output_tokens"].sum(),
    "input_cost_usd": leaderboard["input_cost_usd"].sum(),
    "output_cost_usd": leaderboard["output_cost_usd"].sum(),
    "total_cost_usd": leaderboard["total_cost_usd"].sum(),
    "cost_per_1m_input_usd": pd.NA,
    "cost_per_1m_output_usd": pd.NA,
    "branch": pd.NA,
    "capabilities_file": pd.NA,
    "rounds_used": pd.NA,
}, name="TOTAL (все модели)")

leaderboard_with_total = pd.concat([leaderboard, totals.to_frame().T])
leaderboard_with_total

In [ ]:
# Самоотчёты моделей на последнем раунде (если задача не была решена раньше) —
# отдельно, т.к. это текст, а не число, в основную таблицу не лезет удобно.
github_task_rounds_df[["model_id", "branch", "solved", "rounds_used", "self_report"]]

"## 8. Архивировать сырые результаты (`*.run.json`)\n\n"
"Локальных файлов агента больше нет (модели пишут прямо в реальный "
"репозиторий) — архивируем только `*.run.json`, чтобы можно было потом "
"пересчитать стоимость/токены отдельно от live-прогона."

In [ ]:
def archive_run_json(archive_name: str = None) -> str:
    if archive_name is None:
        archive_name = f"github_benchmark_results_{datetime.now():%Y%m%d_%H%M%S}"
    archive_path = os.path.join(NOTEBOOK_ROOT, f"{archive_name}.zip")

    run_json_files = sorted(set(
        glob.glob(os.path.join(NOTEBOOK_ROOT, "*.run.json"))
        + glob.glob(os.path.join(NOTEBOOK_ROOT, "**", "*.run.json"), recursive=True)
    ))

    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in run_json_files:
            zf.write(path, arcname=os.path.basename(path))

    print(f"Архив готов: {archive_path}")
    print(f"  - *.run.json: {len(run_json_files)}")
    return archive_path


archive_path = archive_run_json()

# ---------------------------------------------------------------- 10. Примечания
cells.append(md_cell(
"## Примечания\n\n"
"- **Токен**: только через Kaggle Secrets (см. раздел 2). Никогда не "
"вставляйте его текстом в ячейку, если планируете сохранять/публиковать "
"версию ноутбука.\n"
"- **Идемпотентность**: `create_branch` и `write_file` безопасно вызывать "
"повторно — ветка не пересоздаётся, файл просто обновляется новым коммитом "
"(нужен `sha` существующего файла — код сам его подтягивает).\n"
"- **Уборка веток**: скрипт ничего не удаляет. Если веток `<модель>` "
"накопится больше, чем нужно (например, после нескольких прогонов с "
"разными наборами моделей), удалите лишние вручную через GitHub UI/API.\n"
"- **Rate limits**: GitHub REST API — 5000 запросов/час на авторизованный "
"токен, при `STEPS_PER_ROUND=6 × MAX_ROUNDS=4` на модель это далеко не "
"проблема при разумном числе моделей.\n"
"- **Бюджет шагов/раундов не общий** — у каждой модели свой, одна модель не "
"может исчерпать бюджет другой (см. обсуждение раньше в этом чате).\n"
"- Если прогон 5+ моделей упрётся в лимиты именно Kaggle Model Proxy "
"(квота/таймаут сессии, а не бюджет шагов) — это отдельная история, "
"смотрите `results.errored_runs` и текст ошибки, чтобы отличить одно от "
"другого."

In [ ]:
import os
import requests
from kaggle_secrets import UserSecretsClient

# ==== настройки — подставь свои ====
GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN") # os.environ.get("GITHUB_TOKEN")  # тот же токен, что использует бот
OWNER = "mlaa4ml"                # владелец репозитория
REPO = "KaggleModelsRepo"        # имя репозитория
BASE_BRANCH = "main"             # ветка, от которой создаём новую

HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

def check(label, resp):
    print(f"\n=== {label} ===")
    print("status:", resp.status_code)
    try:
        print("body:", resp.json())
    except Exception:
        print("body (raw):", resp.text[:500])

# --- 0. Кто мы вообще (какой токен, какие права/scopes видны в заголовках) ---
r = requests.get("https://api.github.com/user", headers=HEADERS)
check("GET /user (идентификация токена)", r)
print("X-OAuth-Scopes:", r.headers.get("X-OAuth-Scopes"))
print("X-Accepted-OAuth-Scopes:", r.headers.get("X-Accepted-OAuth-Scopes"))

# --- 1. Читаем (это уже работает, по твоим логам) ---
r = requests.get(f"https://api.github.com/repos/{OWNER}/{REPO}/branches", headers=HEADERS)
check("GET /branches (чтение)", r)

# --- 2. Проверяем права приложения на конкретный репозиторий (для fine-grained PAT / GitHub App) ---
r = requests.get(f"https://api.github.com/repos/{OWNER}/{REPO}/installation", headers=HEADERS)
check("GET /installation (если это GitHub App)", r)

# --- 3. Пробуем получить SHA базовой ветки (нужно для создания новой ветки) ---
r = requests.get(f"https://api.github.com/repos/{OWNER}/{REPO}/git/ref/heads/{BASE_BRANCH}", headers=HEADERS)
check(f"GET /git/ref/heads/{BASE_BRANCH}", r)
base_sha = None
if r.status_code == 200:
    base_sha = r.json()["object"]["sha"]
    print("base_sha:", base_sha)

# --- 4. САМА ПРОВЕРКА ГИПОТЕЗЫ: создание ветки (write) ---
if base_sha:
    test_branch = "diagnostic-write-test"
    r = requests.post(
        f"https://api.github.com/repos/{OWNER}/{REPO}/git/refs",
        headers=HEADERS,
        json={"ref": f"refs/heads/{test_branch}", "sha": base_sha},
    )
    check("POST /git/refs (создание ветки — write)", r)

    # если получилось — сразу удалим тестовую ветку, чтобы не мусорить
    if r.status_code == 201:
        d = requests.delete(
            f"https://api.github.com/repos/{OWNER}/{REPO}/git/refs/heads/{test_branch}",
            headers=HEADERS,
        )
        print("cleanup delete status:", d.status_code)

# --- 5. Прямая проверка записи файла (PUT contents) — на существующей ветке, без создания новой ---
r = requests.put(
    f"https://api.github.com/repos/{OWNER}/{REPO}/contents/DIAGNOSTIC_TEST.md",
    headers=HEADERS,
    json={
        "message": "diagnostic write test",
        "content": "dGVzdA==",  # base64("test")
        "branch": BASE_BRANCH,
    },
)
check("PUT /contents/DIAGNOSTIC_TEST.md (запись файла — write)", r)

In [ ]:
import json
import glob
import pandas as pd


def load_github_benchmark_costs(results_dir: str = ".", pattern: str = "*.run.json") -> pd.DataFrame:
    """
    Читает все .run.json из results_dir и строит таблицу с usage/cost по моделям.

    Токены и стоимость берутся из conversations[i]['metrics'] — там, где
    conv['metrics'] непуст, это одна "Tool loop"-беседа = один раунд задачи.
    Стоимость в файлах хранится в нанодолларах (1e9 нанодолларов = $1).
    """
    rows = []

    for path in sorted(glob.glob(f"{results_dir}/{pattern}")):
        with open(path) as f:
            data = json.load(f)

        model_id = data["modelVersion"]["slug"]
        state = data["state"]

        input_tokens = 0
        output_tokens = 0
        input_cost = 0.0
        output_cost = 0.0
        latency_ms = 0
        rounds_used = 0

        for conv in data.get("conversations", []):
            m = conv.get("metrics") or {}
            if not m:
                continue  # пустые metrics — это не раунд, а служебная запись беседы
            rounds_used += 1
            input_tokens += m.get("inputTokens", 0) or 0
            output_tokens += m.get("outputTokens", 0) or 0
            input_cost += int(m.get("inputTokensCostNanodollars", 0) or 0) / 1e9
            output_cost += int(m.get("outputTokensCostNanodollars", 0) or 0) / 1e9
            latency_ms += int(m.get("totalBackendLatencyMs", 0) or 0)

        total_cost = input_cost + output_cost
        cost_per_1m_input = (input_cost / input_tokens * 1e6) if input_tokens else None
        cost_per_1m_output = (output_cost / output_tokens * 1e6) if output_tokens else None

        solved = None
        for r in data.get("results", []):
            if r.get("type") == "AGGREGATED":
                solved = r.get("booleanResult")

        rows.append({
            "model_id": model_id,
            "state": state,
            "solved": solved,
            "rounds_used": rounds_used,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "input_cost_usd": round(input_cost, 6),
            "output_cost_usd": round(output_cost, 6),
            "total_cost_usd": round(total_cost, 6),
            "cost_per_1m_input_usd": round(cost_per_1m_input, 3) if cost_per_1m_input else None,
            "cost_per_1m_output_usd": round(cost_per_1m_output, 3) if cost_per_1m_output else None,
            "latency_ms_total": latency_ms,
        })

    df = pd.DataFrame(rows).sort_values("model_id").reset_index(drop=True)

    # строка TOTAL по всем моделям — через .loc, чтобы не ловить FutureWarning от concat
    # с all-NA колонками (state/solved/cost_per_1m у итоговой строки не определены)
    if not df.empty:
        total_idx = len(df)
        df.loc[total_idx, "model_id"] = "TOTAL (все модели)"
        df.loc[total_idx, "rounds_used"] = df["rounds_used"].sum()
        df.loc[total_idx, "input_tokens"] = df["input_tokens"].sum()
        df.loc[total_idx, "output_tokens"] = df["output_tokens"].sum()
        df.loc[total_idx, "input_cost_usd"] = round(df["input_cost_usd"].sum(), 6)
        df.loc[total_idx, "output_cost_usd"] = round(df["output_cost_usd"].sum(), 6)
        df.loc[total_idx, "total_cost_usd"] = round(df["total_cost_usd"].sum(), 6)
        df.loc[total_idx, "latency_ms_total"] = df["latency_ms_total"].sum()
        # state / solved / cost_per_1m_* для итоговой строки осознанно оставляем NaN

    return df


# использование:
costs_df = load_github_benchmark_costs(results_dir="/kaggle/working", pattern="github_repo_capabilities*.run.json")
costs_df